# Chile v2.2 -- Export probability raster for dNBR alignment

Reruns the exact Chile zero-shot inference from `15_train_v22.ipynb` (cell `[23a]`),
but this time exports the reconstructed probability scene as a georeferenced GeoTIFF
instead of discarding it after vectorization. Needed to align against the standalone
dNBR raster (`chile_dnbr_T19HBD.tif`) and compute real IoU / AUC-ROC / Precision /
Recall for Chile (FUTURO-002).

**Environment**: Colab, GPU required (A100 recommended). Run all cells top to bottom.


In [ ]:
# [1] Install dependencies -- IDEMPOTENT, safe to re-run or re-execute via "Run all"
# after the restart below. Only installs and restarts once; if everything is already
# importable it just prints OK and moves on to cell [2] without touching the kernel.
import subprocess, sys, os

needs = False
try:
    import numpy as np
    from terratorch.registry import BACKBONE_REGISTRY
    import rasterio, geopandas, pyproj
    print(f'OK -- numpy={np.__version__}, all deps ready. Continue to cell [2].')
except Exception as e:
    needs = True
    print(f'Installing ({e})...')

if needs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'terratorch', 'einops', 'timm', 'segmentation-models-pytorch',
                    'rasterio', 'geopandas', 'pystac-client', 'planetary-computer',
                    'shapely', 'pyproj'], check=True)
    print('Restarting kernel once -- after it restarts, continue from cell [2] (do NOT re-run this cell manually).')
    os.kill(os.getpid(), 9)


In [ ]:
# [2] Drive mount
from google.colab import drive
drive.mount('/content/drive')
IN_COLAB = True


In [ ]:
# [3] Imports, seed, device
import json, shutil, subprocess, random
from pathlib import Path
from tqdm import tqdm

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import rasterio
from rasterio.transform import from_origin

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

from terratorch.registry import BACKBONE_REGISTRY

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch={torch.__version__}  device={DEVICE}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# [4] Paths and constants
BASE    = Path('/content/drive/MyDrive/GeoAI/wildfire-spread')
CKPT    = BASE / 'models/best_prithvi_v22_burn_scar_wildfire.pth'
RESULTS = BASE / 'results'
OUTPUTS_V22 = BASE / 'outputs' / 'v2.2'
OUTPUTS_V22.mkdir(parents=True, exist_ok=True)

BAND_IDX     = [0, 1, 2, 4, 5, 6]
PRITHVI_MEAN = np.array([0.033349, 0.057012, 0.058897, 0.232325, 0.197285, 0.119449], dtype=np.float32)
PRITHVI_STD  = np.array([0.022691, 0.026808, 0.040041, 0.077917, 0.087087, 0.072420], dtype=np.float32)
IMG_SIZE = 224
T        = (256 - IMG_SIZE) // 2
RES      = 10


In [ ]:
# [5] Architecture -- MultiScaleNeck + FPNDecoder + PrithviSegModelV2
# (copied verbatim from 15_train_v22.ipynb cell [6] -- do not change, avoids
# re-triggering the Prithvi loading errors already documented in prithvi_loading.md)
class MultiScaleNeck(nn.Module):
    def __init__(self, n_side=14, d_model=1024, fpn_ch=256):
        super().__init__()
        self.n_side = n_side
        self.lateral = nn.ModuleList([
            nn.Sequential(nn.Conv2d(d_model, fpn_ch, 1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(4)
        ])
        self.td_conv = nn.ModuleList([
            nn.Sequential(nn.Conv2d(fpn_ch, fpn_ch, 3, padding=1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(3)
        ])
    def tok2map(self, tokens):
        B, L, D = tokens.shape
        return tokens.permute(0, 2, 1).reshape(B, D, self.n_side, self.n_side)
    def forward(self, layers_out):
        feats = [self.lateral[i](self.tok2map(layers_out[idx][:, 1:, :]))
                 for i, idx in enumerate([5, 11, 17, 23])]
        out = feats[3]
        out = self.td_conv[0](feats[2] + out)
        out = self.td_conv[1](feats[1] + out)
        out = self.td_conv[2](feats[0] + out)
        return out


class FPNDecoder(nn.Module):
    def __init__(self, in_ch=256):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 2, stride=2), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256,   128, 2, stride=2), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128,    64, 2, stride=2), nn.BatchNorm2d(64),  nn.GELU(),
            nn.ConvTranspose2d(64,     32, 2, stride=2), nn.BatchNorm2d(32),  nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )
    def forward(self, x): return self.up(x)


class PrithviSegModelV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BACKBONE_REGISTRY.build('prithvi_eo_v2_300')
        self.neck     = MultiScaleNeck(n_side=14, d_model=1024, fpn_ch=256)
        self.decoder  = FPNDecoder(in_ch=256)
    def forward(self, x):
        return self.decoder(self.neck(self.backbone(x.unsqueeze(2))))


In [ ]:
# [6] Load the trained checkpoint (no training here, inference only)
model = PrithviSegModelV2().to(DEVICE)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()
print(f'Checkpoint loaded: {CKPT.name}')


In [ ]:
# [7] Inference + scene reconstruction helpers
# (copied verbatim from 15_train_v22.ipynb cells [16]-[17])
class PatchDataset(Dataset):
    def __init__(self, img_paths):
        self.img_paths = img_paths
    def __len__(self): return len(self.img_paths)
    def __getitem__(self, idx):
        arr = np.load(self.img_paths[idx], mmap_mode='r').astype(np.float32)
        img = arr[BAND_IDX] / 10000.0
        img = (img - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
        img = img[:, T:T+IMG_SIZE, T:T+IMG_SIZE]
        return torch.from_numpy(img)


def run_inference(img_paths, batch_size=8, num_workers=2):
    ds     = PatchDataset(img_paths)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
    probs_list = []
    with torch.no_grad():
        for imgs in tqdm(loader, desc='Inference'):
            probs = model(imgs.to(DEVICE)).sigmoid().squeeze(1).cpu().numpy()
            for b in range(len(imgs)):
                probs_list.append(probs[b])
    return probs_list


def parse_row_col(path):
    stem  = Path(path).stem
    parts = stem.split('_r')
    rc    = parts[-1]
    return int(rc.split('_c')[0]), int(rc.split('_c')[1])


def reconstruct_scene(img_paths, probs_list, patch_size=IMG_SIZE, crop_offset=T):
    rows_cols = []
    for p in img_paths:
        r, c = parse_row_col(p)
        rows_cols.append((r + crop_offset, c + crop_offset))

    max_row = max(r for r, c in rows_cols) + patch_size
    max_col = max(c for r, c in rows_cols) + patch_size

    scene_sum   = np.zeros((max_row, max_col), dtype=np.float32)
    scene_count = np.zeros((max_row, max_col), dtype=np.float32)

    for (row, col), prob in zip(rows_cols, probs_list):
        scene_sum  [row:row+patch_size, col:col+patch_size] += prob
        scene_count[row:row+patch_size, col:col+patch_size] += 1

    has_data   = scene_count > 0
    scene_prob = np.where(has_data, scene_sum / np.maximum(scene_count, 1), np.nan)
    return scene_prob, max_row, max_col, rows_cols

print('Helpers ready.')


In [ ]:
# [8] Chile ZS reconstruction + export as georeferenced GeoTIFF (the new part)
data_dir = BASE / 'data' / 'zs' / 'chile' / 'patches'
ch_img_paths = sorted(data_dir.glob('*.npy'))
print(f'Patches: {len(ch_img_paths)}')
assert ch_img_paths, f'No patches found in {data_dir}'

manifest_path = data_dir.parent / 'manifest.json'
with open(manifest_path) as f:
    manifest = json.load(f)
best_entry = max(manifest['tiles'], key=lambda t: t['fire_patches'] / t['patches'])
best_tile  = best_entry['tile'].lstrip('T')
print(f'Most burned tile: {best_tile} ({best_entry["fire_patches"]}/{best_entry["patches"]} fire patches)')

ch_img_tile = [p for p in ch_img_paths if best_tile in p.stem] or ch_img_paths
ch_ox   = best_entry['transform'][2]
ch_oy   = best_entry['transform'][5]
ch_epsg = int(best_entry['crs'].split(':')[1])

ch_probs_all  = run_inference(ch_img_paths)
tile_indices  = [i for i, p in enumerate(ch_img_paths) if best_tile in p.stem]
ch_probs_tile = [ch_probs_all[i] for i in tile_indices] if tile_indices else ch_probs_all

ch_scene, _, _, _ = reconstruct_scene(ch_img_tile, ch_probs_tile)
print(f'Scene shape: {ch_scene.shape}')  # should be (10864, 10864), matching the v2.2 run

# --- export ---
transform = from_origin(ch_ox, ch_oy, RES, RES)
profile = {
    'driver': 'GTiff', 'dtype': 'float32', 'nodata': np.nan,
    'width': ch_scene.shape[1], 'height': ch_scene.shape[0],
    'count': 1, 'crs': f'EPSG:{ch_epsg}', 'transform': transform,
    'compress': 'lzw',
}

local_tif = Path('/content/chile_prob_scene_v22.tif')
with rasterio.open(local_tif, 'w', **profile) as dst:
    dst.write(ch_scene.astype('float32'), 1)

dst_tif = OUTPUTS_V22 / 'chile_prob_scene_v22.tif'
shutil.copy(local_tif, dst_tif)
assert dst_tif.stat().st_size > 1_000_000, f'Suspiciously small file: {dst_tif}'
print(f'Saved: {dst_tif} ({dst_tif.stat().st_size / 1e6:.1f} MB)')

with rasterio.open(dst_tif) as check:
    print(f'CRS: {check.crs}')
    print(f'Bounds: {check.bounds}')
    print(f'Shape: {check.shape}')

print('DONE. This file is what the local dNBR alignment step needs next.')
